## Phase 3: Feature engineering

In [3]:
import numpy as np
import pandas as pd

# --- re-read the full log (needed for lifetime features) ---
ratings = pd.read_csv("../data/ml-25m/ratings.csv",
                      dtype={"userId": "int32", "movieId": "int32",
                             "rating": "float32", "timestamp": "int64"})
ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")

# --- load what notebook 01 produced ---
hist_tr = pd.read_parquet("../data/history_train.parquet")
hist_te = pd.read_parquet("../data/history_test.parquet")
y_tr    = pd.read_parquet("../data/labels_train.parquet")["lapsed"]
y_te    = pd.read_parquet("../data/labels_test.parquet")["lapsed"]

# --- anchors are just constants; restate them ---
anchor_tr = pd.Timestamp("2018-01-01")
anchor_te = pd.Timestamp("2019-01-01")

print(len(y_tr), len(y_te), y_tr.mean().round(3), y_te.mean().round(3))

5045 5160 0.503 0.511


In [4]:
def build_features(history, ratings, anchor, users):
    """Turn one history window into one row per user.

    history : ratings inside the feature window, eligible users only
    ratings : the full log (used only for lifetime facts before the anchor)
    users   : the eligible user ids, in the same order as the labels
    """
    anchor = pd.Timestamp(anchor)
    h = history.copy()

    # How long before the anchor did each event happen?
    h["days_before"] = (anchor - h["date"]).dt.days
    h["day"] = h["date"].dt.normalize()          # for counting distinct active days

    g = h.groupby("userId")
    f = pd.DataFrame(index=pd.Index(users, name="userId"))

    # --- recency: how long since we last saw them ---
    f["recency_days"] = g["days_before"].min()

    # --- frequency at several scales ---
    f["n_365"] = g.size()
    f["n_90"]  = h[h.days_before <= 90].groupby("userId").size()
    f["n_30"]  = h[h.days_before <= 30].groupby("userId").size()
    f["n_7"]   = h[h.days_before <=  7].groupby("userId").size()
    f[["n_90", "n_30", "n_7"]] = f[["n_90", "n_30", "n_7"]].fillna(0)

    # --- breadth and intensity ---
    f["n_movies"]       = g["movieId"].nunique()
    f["n_active_days"]  = g["day"].nunique()
    f["per_active_day"] = f["n_365"] / f["n_active_days"]

    # --- taste ---
    f["rating_mean"] = g["rating"].mean()
    f["rating_std"]  = g["rating"].std().fillna(0)

    # --- trend: is recent activity above or below their 90-day rate? ---
    f["trend_30_90"] = f["n_30"] / f["n_90"].clip(lower=1)

    # --- lifetime facts, from everything before the anchor ---
    past = ratings[ratings.date < anchor]
    f["tenure_days"]      = (anchor - past.groupby("userId")["date"].min()
                             .reindex(f.index)).dt.days
    f["lifetime_ratings"] = past.groupby("userId").size().reindex(f.index)

    return f

In [5]:
X_tr = build_features(hist_tr, ratings, anchor_tr, y_tr.index)
X_te = build_features(hist_te, ratings, anchor_te, y_te.index)

print(X_tr.shape, X_te.shape)
print("\nany missing?\n", X_tr.isna().sum().loc[lambda s: s > 0])
X_tr.describe().round(2).T

(5045, 13) (5160, 13)

any missing?
 Series([], dtype: int64)


,count,mean,std,min,25%,50%,75%,max
recency_days,5045.0,31.69,27.63,0.00,6.00,24.00,56.00,89.00
n_365,5045.0,157.04,259.17,1.00,33.00,73.00,170.00,4802.00
n_90,5045.0,69.70,155.93,1.00,7.00,24.00,63.00,3348.00
n_30,5045.0,23.24,95.66,0.00,0.00,1.00,10.00,3015.00
n_7,5045.0,7.25,51.00,0.00,0.00,0.00,1.00,2075.00
n_movies,5045.0,157.04,259.17,1.00,33.00,73.00,170.00,4802.00
n_active_days,5045.0,15.85,25.58,1.00,2.00,6.00,19.00,331.00
per_active_day,5045.0,35.53,73.37,1.00,3.00,12.74,38.00,1674.00
rating_mean,5045.0,3.69,0.57,0.75,3.34,3.72,4.07,5.00
rating_std,5045.0,0.81,0.31,0.00,0.60,0.78,0.99,2.47


In [6]:
# Sanity check: does each feature actually relate to lapsing?
tr = X_tr.copy()
tr["lapsed"] = y_tr

for c in ["recency_days", "n_90", "tenure_days", "trend_30_90", "n_movies"]:
    q = pd.qcut(tr[c], 5, duplicates="drop")
    print(f"\n{c}")
    print(tr.groupby(q, observed=True)["lapsed"].agg(["mean", "size"]).round(3).to_string())


recency_days
                mean  size
recency_days              
(-0.001, 4.0]  0.193  1072
(4.0, 15.0]    0.345   989
(15.0, 36.0]   0.533  1015
(36.0, 61.0]   0.674   963
(61.0, 89.0]   0.793  1006

n_90
                 mean  size
n_90                       
(0.999, 5.0]    0.500  1039
(5.0, 15.0]     0.308   987
(15.0, 34.0]    0.488  1024
(34.0, 83.0]    0.602   995
(83.0, 3348.0]  0.613  1000

tenure_days
                   mean  size
tenure_days                  
(-0.001, 60.0]    0.778  1014
(60.0, 177.8]     0.713  1004
(177.8, 589.0]    0.382  1012
(589.0, 1570.0]   0.338  1007
(1570.0, 7412.0]  0.302  1008

trend_30_90
                  mean  size
trend_30_90                 
(-0.001, 0.288]  0.576  3028
(0.288, 1.0]     0.392  2017

n_movies
                  mean  size
n_movies                    
(0.999, 28.0]    0.606  1042
(28.0, 53.0]     0.575   978
(53.0, 101.0]    0.495  1017
(101.0, 215.0]   0.469  1001
(215.0, 4802.0]  0.368  1007


In [7]:
# Is high activity actually a proxy for being new?
print(tr.groupby(pd.qcut(tr.n_90, 5, duplicates="drop"), observed=True)
      .agg(lapse=("lapsed", "mean"),
           median_tenure=("tenure_days", "median"),
           n=("lapsed", "size")).round(1).to_string())

                lapse  median_tenure     n
n_90                                      
(0.999, 5.0]      0.5          783.0  1039
(5.0, 15.0]       0.3          788.0   987
(15.0, 34.0]      0.5          321.5  1024
(34.0, 83.0]      0.6           87.0   995
(83.0, 3348.0]    0.6           62.0  1000


In [8]:
# Save for modeling notebook
X_tr.to_parquet("../data/features_train.parquet")
X_te.to_parquet("../data/features_test.parquet")
print("saved")

saved
